In [4]:
from langgraph.graph import StateGraph ,START,END
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
from typing import TypedDict , Annotated
from dotenv import load_dotenv
from pydantic import Field , BaseModel
import operator
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

In [6]:
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

In [ ]:
load_dotenv()  # Load environment variables from .env file
class upscevalschema(BaseModel):
    feedback:str=Field(description="detailed feedback for the essay")
    score:int = Field(description="score out of 10")


llm = HuggingFaceEndpoint(
repo_id="Qwen/Qwen2.5-7B-Instruct",
task="text-generation",
provider="featherless-ai",
max_new_tokens=1024
) # type: ignore

model = ChatHuggingFace(llm=llm)
# model_with_str_output=model.with_structured_output(upscevalschema)
parser=PydanticOutputParser(pydantic_object=upscevalschema)


feedback="The essay provides a comprehensive overview of the challenges and opportunities that India faces in the age of AI. It highlights the country's strengths and specific areas where AI can make a significant impact, such as agriculture, healthcare, and education. However, the essay could benefit from more concrete examples and statistics to support its arguments. Additionally, the discussion on challenges could be more balanced by providing more on the positive aspects of addressing these issues. The conclusion ties the essay together well, emphasizing the importance of responsible AI adoption for India's future. The writing is clear and engaging, making the essay accessible to a wide audience. However, it could be improved by aligning more closely with the specific requirements of the UPSC essay format, such as addressing the question more directly and ensuring a structured argument." score=8


In [17]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [ ]:
def language_feedback(state:UPSCState):
    essay=state['essay']
    prompt = PromptTemplate(
        template="Evaluate the language quality of the following\nEssay: {essay}and provide a feedback and assign a score out of 10  \n{format_instructions}\n",
        input_variables=["essay"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
        
    )
    chain= prompt | model | parser
    lan_feedback=chain.invoke({'essay':essay})
    return {'language_feedback': lan_feedback.feedback, 'individual_scores': [lan_feedback.score]}

def analysis_feedback(state:UPSCState):
    essay=state['essay']
    prompt = PromptTemplate(
        template="Evaluate the depth analysis of the essay \nEssay: {essay}and provide a feedback and assign a score out of 10  \n{format_instructions}\n",
        input_variables=["essay"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
        
    )
    chain= prompt | model | parser
    lan_feedback=chain.invoke({'essay':essay})
    return {'analysis_feedback': lan_feedback.feedback, 'individual_scores': [lan_feedback.score]}

def clarity_feedback(state:UPSCState):
    essay=state['essay']
    prompt = PromptTemplate(
        template="Evaluate the clarity of thought of the following essay \nEssay: {essay} and provide a feedback and assign a score out of 10 \n{format_instructions}\n",
        input_variables=["essay"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
        
    )
    chain= prompt | model | parser
    lan_feedback=chain.invoke({'essay':essay})
    return {'analysis_feedback': lan_feedback.feedback, 'individual_scores': [lan_feedback.score]}